# 05 - Run Experiments

5x2 cross-validation for the linear, feedforward, and autoencoder networks on each data set.

In [1]:
import pandas as pd
import numpy as np

import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.preprocessing import load_processed, prepare_xy
from src.neural_network import NeuralNetwork
from src.linear_network import LogisticRegression, LinearRegression
from src.evaluation import cross_validation_loop
from src.tuning import holdout_split, random_search

In [3]:
seed = 0

In [4]:
# Breast Cancer Dataset
data, meta = load_processed("../data/processed/breast_cancer_processed.csv")
X, y, y_labels = prepare_xy(data, meta)

# 20% tuning set, 80% for the 5x2 CV (same split for every model)
cv_idx, val_idx = holdout_split(len(X), y_labels, 0.2, seed=seed)
X_cv, y_cv, yl_cv = X[cv_idx], y[cv_idx], y_labels[cv_idx]
X_val, y_val, yl_val = X[val_idx], y[val_idx], y_labels[val_idx]

# ---------------- Logistic Regression ----------------
print("Logistic Regression")
log_space = {"learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": [16, 32, None]}
log_factory = lambda c, s: LogisticRegression(X.shape[1], y.shape[1], c["learning_rate"], seed=s)

best_log, log_trials = random_search(
    log_factory, log_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=15, max_epochs=1000, checkpoint_every=50, seed=seed,
    log_path="../results/tuning/breast_cancer_logistic.csv")
print("Best:", best_log)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: log_factory(best_log, seed),
    epochs=best_log["epochs"], batch_size=best_log["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}\n")

# ---------------- Neural Network ----------------
print("Neural Network")
nn_space = {"hidden1": [8, 16, 32], "hidden2": [4, 8, 16],
            "learning_rate": ("log_uniform", 0.001, 1.0), "batch_size": [16, 32, None]}
nn_factory = lambda c, s: NeuralNetwork(
    [X.shape[1], c["hidden1"], c["hidden2"], y.shape[1]],
    mode="classification", learning_rate=c["learning_rate"], seed=s)

best_nn, nn_trials = random_search(
    nn_factory, nn_space, X_cv, y_cv, X_val, y_val, yl_val, mode="classification",
    n_trials=30, max_epochs=1000, checkpoint_every=50,
    constraint=lambda c: c["hidden2"] <= c["hidden1"], seed=seed,
    log_path="../results/tuning/breast_cancer_nn.csv")
print("Best:", best_nn)

results = cross_validation_loop(
    X_cv, y_cv, yl_cv, lambda: nn_factory(best_nn, seed),
    epochs=best_nn["epochs"], batch_size=best_nn["batch_size"], seed=seed)
print(f"Mean Classification Error: {np.mean(results['error'])} \nMean Cross Entropy Loss: {np.mean(results['cross_entropy'])}")

Logistic Regression
trial   1/15  val_loss=0.0965  epochs=50  {'learning_rate': 0.081448869730761, 'batch_size': 32}
trial   2/15  val_loss=0.0963  epochs=1000  {'learning_rate': 0.0013271517125308802, 'batch_size': 16}
trial   3/15  val_loss=0.0966  epochs=1000  {'learning_rate': 0.0011209413928285786, 'batch_size': 16}
trial   4/15  val_loss=0.0960  epochs=100  {'learning_rate': 0.5473520283788641, 'batch_size': None}
trial   5/15  val_loss=0.0963  epochs=850  {'learning_rate': 0.06605524798050313, 'batch_size': None}
trial   6/15  val_loss=0.0965  epochs=1000  {'learning_rate': 0.04274398161693902, 'batch_size': None}
trial   7/15  val_loss=0.0962  epochs=150  {'learning_rate': 0.6385828802382952, 'batch_size': 16}
trial   8/15  val_loss=0.2054  epochs=1000  {'learning_rate': 0.0010190969469382445, 'batch_size': None}
trial   9/15  val_loss=0.0968  epochs=50  {'learning_rate': 0.37343311708676946, 'batch_size': 32}
trial  10/15  val_loss=0.0972  epochs=500  {'learning_rate': 0.15451